# Sesión 04 - Lab 1: Ingesta JDBC desde una base transaccional

Este laboratorio construye un pipeline de ingesta JDBC de punta a punta contra la Azure SQL Database real de esta sesión, con el sample **AdventureWorksLT** (schema `SalesLT`) — el workspace se eleva puntualmente a Premium para esta demo, porque Free Edition restringe por defecto el acceso saliente que JDBC necesita (ver celda de Prerrequisitos). Lab 1A y Lab 1B muestran dos formas de abrir la misma conexión (Secret Scope clásico vs. SQL Server Connection de Unity Catalog); Lab 1C y Lab 1D reutilizan el resultado real de Lab 1B para poblar y extender la tabla Bronze, sin ningún archivo simulado de por medio.

## Prerrequisitos para Lab 1A/1B (los hace el docente una sola vez, antes de la sesión)

1. **Elevar el workspace a Premium** para esta sesión — Free Edition restringe por defecto el acceso saliente a un host externo como una Azure SQL Database.
2. **Crear la Azure SQL Database con el sample AdventureWorksLT:**
   - Azure Portal → *Create a resource* → *SQL Database* → crear un servidor lógico nuevo (autenticación SQL, admin login/password).
   - En la pestaña *Additional settings* → *Data source*, seleccionar **Sample** — esto crea el schema `SalesLT` con datos de ejemplo (`SalesLT.Customer`, entre otras tablas).
   - Tier recomendado para una demo de aula: `Basic` o `General Purpose Serverless`, el más económico disponible.
3. **Firewall del servidor Azure SQL:**
   - *Networking* → agregar tu propia IP (para administrar la base desde tu computadora).
   - Agregar las IPs de salida del compute serverless del workspace de Databricks — ver "Configure a firewall for serverless compute access" en Microsoft Learn. No usar *Allow Azure services* como sustituto: es una regla amplia que acepta tráfico desde cualquier recurso de Azure, no solo desde este workspace.
4. **Usuario de ingesta dedicado, sin privilegios de administrador** — mismo antipatrón de credenciales marcado desde la Sesión 01, ahora aplicado a "nunca conectar un pipeline con la cuenta admin/sa":
   ```sql
   CREATE LOGIN ingesta_databricks WITH PASSWORD = '<password-fuerte>';
   CREATE USER ingesta_databricks FOR LOGIN ingesta_databricks;
   GRANT SELECT ON SCHEMA::SalesLT TO ingesta_databricks;
   ```
5. **Secret Scope con las credenciales de `ingesta_databricks`** (Lab 1A) — desde la Databricks CLI, no desde el notebook.
6. **Driver JDBC de Microsoft para SQL Server** (`mssql-jdbc-<version>.jre11.jar`, descargado del sitio oficial del driver) subido a `/Volumes/dbassociate/default/vol_landing/sesion_04/drivers/` — lo usa Lab 1A. Lab 1B no lo necesita: la SQL Server Connection de Unity Catalog trae su propio conector nativo.

**Importante:** `SalesLT.Customer` trae columnas `PasswordHash`/`PasswordSalt` que no queremos aterrizar en Bronze. Lab 1A/1B proyectan explícitamente solo las columnas de negocio — nunca un `SELECT *` ciego sobre una tabla real.

## Verificación del entorno

In [0]:
# Probamos un secret creado
test = dbutils.secrets.get(scope="scope-demo", key="secret-test")

In [0]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_04")

## Lab 1A: JDBC clásico con Secret Scope

Patrón genérico para leer cualquier base de datos relacional desde un notebook, sin depender de un securable de Unity Catalog. Corre en vivo contra la Azure SQL Database de esta sesión (ver Prerrequisitos) — reemplazá `<tu-servidor>` y `<tu-base-de-datos>` por los valores reales antes de ejecutar.

**Sobre `dbtable`:** acepta una subconsulta entre paréntesis — el filtro se ejecuta en el motor de origen (predicate pushdown), no después de traer toda la tabla a Spark. Acá se usa además para proyectar solo columnas de negocio, dejando afuera `PasswordHash`/`PasswordSalt`.

**Antipatrón:** nunca escribir usuario/contraseña directo en una celda, ni siquiera en un notebook de prueba — la misma regla de credenciales hardcodeadas marcada desde la Sesión 01.

In [0]:
# En la variable secret debemos guardar datos sensibles.
username = dbutils.secrets.get(scope="jdbc", key="secret-azuresql-username")
password = dbutils.secrets.get(scope="jdbc", key="secret-azuresql-password")

columnas_negocio = "CustomerID, Title, FirstName, MiddleName, LastName, CompanyName, SalesPerson, EmailAddress, Phone, ModifiedDate"

# forma tradicional mediante el driver
df_lab1a = (
    spark.read.format("jdbc")
    # Cadena de conexión
    # .option("url", "jdbc:sqlserver://<tu-servidor>.database.windows.net:1433;database=<tu-base-de-datos>;encrypt=true;trustServerCertificate=false;loginTimeout=30;")
    .option("url", "jdbc:sqlserver://analyticsdmc.database.windows.net:1433;database=AdventureWorks;encrypt=true;trustServerCertificate=false;loginTimeout=30;")
    .option("dbtable", f"(SELECT {columnas_negocio} FROM SalesLT.Customer) AS extract")
    # pasamos los secretos
    .option("user", username)
    .option("password", password)
    .option("fetchsize", "1000")
    .load()
)

df_lab1a.printSchema()
df_lab1a.show(5, truncate=False) # modo texto.

In [0]:
# modo tabular
# display es un action, pues desencadena todas las acciones (action) del dataframe para que se pueda mostrar,
# es decir, ejecuta todos los métodos anterires.
display(
    df_lab1a
)

## Lab 1B: Conexión gobernada — SQL Server Connection de Unity Catalog (Lakehouse Federation)

Alternativa a Lab 1A que no depende de un driver JDBC propio ni de Secret Scope: Unity Catalog se conecta directo a la Azure SQL Database con un conector nativo para SQL Server, sin subir ningún `.jar`. La credencial queda oculta del usuario que consulta, y el gobierno queda a nivel de tabla (más fino que una conexión JDBC genérica, que solo gobierna a nivel de conexión completa).

**1. Crear la Connection y el Foreign Catalog** (una sola vez, desde Catalog Explorer — no hace falta código):

1. **Catalog** → ➕ **Add** → **Create a connection**.
2. **Connection name**: `conn_erp_sqlserver`. **Connection type**: `SQL Server`. **Auth type**: `Username and password`.
3. **Host**: `<tu-servidor>.database.windows.net`. **Port**: `1433`. **User**/**Password**: los del usuario `ingesta_databricks` creado en Prerrequisitos.
4. En la misma wizard, en **Catalog basics**: nombre para el foreign catalog, por ejemplo `erp_sqlserver_federado`. Completá **Access**, **Owner** y **Privileges** según corresponda, y hacé clic en **Save**.
5. La conexión y el foreign catalog quedan creados — `erp_sqlserver_federado.SalesLT.Customer` ya es una tabla más de Unity Catalog.

**2. Consultar el foreign catalog desde el notebook** — se lee como cualquier tabla de Unity Catalog, sin `remote_query()` ni opciones de JDBC. El filtro y la proyección de columnas se empujan automáticamente al motor de origen (pushdown), igual que con `dbtable`/`query` en Lab 1A.

**Ventaja sobre el Lab 1A:** no hay que administrar un driver JDBC ni un Secret Scope propio; el `GRANT`/`REVOKE` se puede otorgar tabla por tabla sobre el foreign catalog (`GRANT SELECT ON TABLE erp_sqlserver_federado.SalesLT.Customer TO ...`), no solo a nivel de toda la conexión. Se retoma en la Sesión 12 (Governance and Security): tanto la Connection como el foreign catalog son securables de Unity Catalog.

In [0]:
# Usamos el catalogo para conectarnos y obtener la data.
df_lab1b = spark.sql(f"""
    SELECT {columnas_negocio}
    FROM erp_sqlserver_federado.SalesLT.Customer
""")

df_lab1b.show(5, truncate=False)

## Lab 1C: Aterrizar el resultado de Lab 1B en Bronze

`df_lab1b` — el resultado real que trajo Lab 1B contra el foreign catalog `erp_sqlserver_federado.SalesLT.Customer` — se aterriza directo en Bronze, sin pasar por ningún archivo intermedio. Mismo criterio de columnas de auditoría usado desde la Sesión 01.

In [0]:
from datetime import datetime
from pyspark.sql.functions import col, current_timestamp, lit

# withColumn es un transformation, no desencadena la acción, no lo activa, solo valida que esas expresiones no fallen.
df_snapshot = (
    df_lab1b
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("erp_clientes_jdbc"))
    .withColumn("batch_id", lit("carga_" + datetime.now().strftime("%Y%m%d_%H%M")))
)

# No es necesario aplicar el display en cada cambio que hagamos porque esto ejecuta todo y lo muestra ocupando memoria, solo con un transformation
# display(df_snapshot)

# guardamos como tabla delta
df_snapshot.write.mode("overwrite").saveAsTable("dbassociate.default.clientes_lab1")

print("Filas cargadas en el snapshot inicial:", df_snapshot.count())

## Lab 1D: Extracción incremental — segunda consulta real al foreign catalog

Vuelve a consultar `erp_sqlserver_federado.SalesLT.Customer` y se queda solo con los clientes cuyo `ModifiedDate` es posterior al último valor ya cargado en Bronze — mismo patrón de pushdown que Lab 1A/1B, ahora aplicado a una segunda llamada real en vez de un archivo estático.

Nota de diseño: contra un sample estático como AdventureWorksLT, esta segunda consulta normalmente trae 0 filas nuevas — es el comportamiento correcto de una extracción incremental idempotente, no un error. Si el origen sí tuviera cambios entre ambas consultas, esta tabla Bronze queda **append-only**, sin resolver duplicados: el cliente actualizado aparecería dos veces (versión original y versión nueva). Resolver cuál fila es la vigente (deduplicación, SCD Type 2) es tema de las Sesiones 05-07, no de Bronze.

In [0]:
ultima_fecha_procesada = spark.sql(
    "SELECT MAX(ModifiedDate) AS ultima FROM dbassociate.default.clientes_lab1"
).first()["ultima"]

# ultima fecha de procesamiento de la tabla SQL server
print("Última fecha ya procesada en la tabla:", ultima_fecha_procesada)

df_incremental = (
    # f string es para interpolación de variables
    # lit es para convertir a lit el valor que le pasemos
    # current_timestamp es para obtener el timestamp actual
    # filter es para filtrar por la columna que le pasemos
    # spark.sql es para ejecutar un query en formato SQL.
    spark.sql(f"""
        SELECT {columnas_negocio}
        FROM erp_sqlserver_federado.SalesLT.Customer
    """)
    .filter(col("ModifiedDate") > lit(ultima_fecha_procesada))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("erp_clientes_jdbc"))
    .withColumn("batch_id", lit("carga_" + datetime.now().strftime("%Y%m%d_%H%M")))
)

# guardamos o ingestamos la data nueva que viene de la base de datos a nuestra tabla delta.
df_incremental.write.mode("append").saveAsTable("dbassociate.default.clientes_lab1")

print("Filas agregadas en la carga incremental:", df_incremental.count())

## Lab 1E: Verificar si la carga incremental dejó clientes con más de una versión

Cuenta cuántas filas tiene cada `CustomerID` en Bronze. Contra AdventureWorksLT, sin cambios reales entre las dos consultas (Lab 1C/1D), lo esperable es que esta consulta no devuelva filas — mismo comportamiento idempotente explicado en Lab 1D.

In [0]:
spark.sql("""
    SELECT CustomerID, COUNT(*) AS versiones
    FROM dbassociate.default.clientes_lab1
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
    ORDER BY versiones DESC, CustomerID
""").show(truncate=False)

## Lab 1F: Observabilidad y errores comunes de ingesta JDBC

No hay un `DESCRIBE HISTORY` específico de JDBC (a diferencia de `COPY INTO`): la observabilidad depende de los logs del propio driver/conector y del job que orquesta la extracción. Errores frecuentes:

- **Driver JAR no encontrado o versión incompatible (Lab 1A)**: falla con `ClassNotFoundException` al abrir la conexión — verificar que el JAR está en el Volume y fue agregado al cluster/compute.
- **Firewall del origen bloqueando la IP de salida de Databricks**: la conexión hace timeout — requiere abrir el firewall de la base de datos hacia las IPs de salida del workspace. Aplica a Lab 1A y a Lab 1B por igual.
- **Credenciales rotadas o expiradas**: falla de autenticación — con Secret Scope (Lab 1A) alcanza con actualizar el secreto; con la SQL Server Connection (Lab 1B), se edita la conexión desde Catalog Explorer, sin tocar el código del notebook.
- **`fetchsize` muy bajo en tablas grandes (Lab 1A)**: lecturas lentas por exceso de round-trips de red — subir el valor reduce viajes de red a costa de más memoria por lote. No aplica a Lab 1B, que no expone esa opción.
- **`SELECT *` sobre una tabla de producción**: trae a Bronze columnas que nunca deberían salir de la fuente (en `SalesLT.Customer`, `PasswordHash`/`PasswordSalt`) — proyectar siempre las columnas de negocio, como hacen Lab 1A/1B.

## Consulta de validación

In [0]:
spark.sql("""
    SELECT CompanyName, SalesPerson, COUNT(*) AS num_clientes
    FROM dbassociate.default.clientes_lab1
    GROUP BY CompanyName, SalesPerson
    ORDER BY num_clientes DESC
""").show(truncate=False)

## Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.default.clientes_lab1")

print("Tabla temporal de este laboratorio eliminada.")